<a href="https://colab.research.google.com/github/ialimustufa/python-crash-course/blob/master/notebooks/07_Fundamentals_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 · Fundamentals capstone

| | |
|---|---|
| Time | about 60 minutes |
| Prerequisites | Notebooks 00–06 |
| Environment | Google Colab, or any Python 3 notebook |
| Thread | Bike-share trips in Indian cities — the same story builds toward the capstone |

## How to use this notebook

1. Click **Open in Colab**, then use **File → Save a copy in Drive**.
2. Run cells **from top to bottom**. If names go missing, use **Runtime → Restart session** and **Run all**.
3. When you see **Predict**, decide the output before you run.
4. Practice cells include a working answer so **Run all** never fails (needed for class and CI). To learn, comment out the answer, rewrite it, then run the check.
5. Hints sit under each practice. Clean solutions are at the end.

Concept sections include diagrams of the idea, not only the code. They show as pictures in Colab and as Mermaid on GitHub.

## What you will be able to do

- Parse a dirty CSV of bike-share trips
- Drop missing, malformed, and duplicate rows and report counts
- Derive features and a data-quality summary
- Split, scale with train-only bounds, and assert invariants


## Capstone: prepare a bike-share table

You are given raw trip logs as CSV text (no download). Produce a training table a later ML library could consume.

**Done means all checks in this notebook print `passed`.** You will not fit a model.

Suggested steps:

1. Parse CSV to dictionaries.
2. Validate and convert types.
3. Remove duplicates, missing required values, and malformed numbers.
4. Derive `km_per_hour` and `age_group`.
5. Summarize quality.
6. Drop the target from features.
7. Deterministic 80/20 split by `id`.
8. Min-max scale `hours`, `km`, and `km_per_hour` using **train** mins and maxes only.

What should happen to each raw row:

| id | Problem | Fate |
|---|---|---|
| 1 | clean | keep |
| 2 | blank `hours` | drop missing |
| 3 | clean | keep |
| 4 | appears twice | keep first, drop duplicate |
| 5 | `hours` is `abc` | drop invalid |
| 6 | clean | keep |
| 7 | clean (teen rider) | keep |
| 8 | clean | keep |

Nine data rows in, six useful rows out. Sorted ids `[1, 3, 4, 6, 7, 8]`; 80% train is the first four.

![Capstone pipeline: parse, clean, derive, split, scale, export](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgY3N2VGV4dFsiQ1NWIHRleHQmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXSAtLT4gcGFyc2VSb3dzWyJwYXJzZSZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgcGFyc2VSb3dzIC0tPiBkcm9wQmFkWyJkcm9wIG1pc3NpbmcsIGludmFsaWQsIGR1cGxpY2F0ZSZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgZHJvcEJhZCAtLT4gZGVyaXZlWyJkZXJpdmUga21fcGVyX2hvdXIgYW5kIGFnZV9ncm91cCZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgZGVyaXZlIC0tPiBzcGxpdERhdGFbInNwbGl0IDgwLzIwIGJ5IGlkJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICBzcGxpdERhdGEgLS0-IHNjYWxlWyJ0cmFpbi1vbmx5IG1pbi1tYXgmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXQogIHNjYWxlIC0tPiBidW5kbGVbIkpTT04gYnVuZGxlJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0=)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart TB
  csvText[CSV text] --> parseRows[parse]
  parseRows --> dropBad["drop missing, invalid, duplicate"]
  dropBad --> derive[derive km_per_hour and age_group]
  derive --> splitData["split 80/20 by id"]
  splitData --> scale[train-only min-max]
  scale --> bundle[JSON bundle]
```

</details>

![Nine rows in, six kept, three dropped for distinct reasons](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgbmluZVsiOSBkYXRhIHJvd3MmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsmbmJzcDsiXSAtLT4ga2VlcFsia2VlcCAxLCAzLCA0LCA2LCA3LCA4Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICBuaW5lIC0tPiBtaXNzWyJkcm9wIGlkIDIgbWlzc2luZyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyZuYnNwOyJdCiAgbmluZSAtLT4gZHVwWyJkcm9wIHNlY29uZCBpZCA0Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0KICBuaW5lIC0tPiBpbnZbImRyb3AgaWQgNSBpbnZhbGlkJm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Jm5ic3A7Il0=)

<details>
<summary>Diagram source</summary>

```mermaid
flowchart TB
  nine[9 data rows] --> keep["keep 1, 3, 4, 6, 7, 8"]
  nine --> miss["drop id 2 missing"]
  nine --> dup["drop second id 4"]
  nine --> inv["drop id 5 invalid"]
```

</details>


## The raw file


In [ ]:
from pathlib import Path
import csv
import io

RAW_CSV = """id,city,hours,km,rider_age,pass_type,returned
1,Mumbai,2,8.5,24,annual,yes
2,Pune,,3.0,31,casual,yes
3,Mumbai,1,4.0,19,annual,no
4,Bengaluru,3,12.0,41,annual,yes
4,Bengaluru,3,12.0,41,annual,yes
5,Mumbai,abc,5.0,22,casual,no
6,Pune,2.5,6.0,28,annual,yes
7,Bengaluru,0.5,1.2,17,casual,yes
8,Mumbai,4,18.0,35,annual,no
"""

DATA_DIR = Path.cwd() / "_course_data"
DATA_DIR.mkdir(exist_ok=True)
raw_path = DATA_DIR / "bike_trips_raw.csv"
raw_path.write_text(RAW_CSV, encoding="utf-8")
print(raw_path.read_text(encoding="utf-8"))


## Practice 07.1 — parse, clean, derive


In [ ]:
# Practice 07.1
def parse_float(text):
    try:
        return float(text)
    except (TypeError, ValueError):
        return None


def age_group(age):
    if age < 18:
        return "teen"
    if age < 35:
        return "adult"
    return "older"


def prepare(csv_text):
    report = {
        "rows_in": 0,
        "dropped_missing": 0,
        "dropped_invalid": 0,
        "dropped_duplicate": 0,
        "rows_out": 0,
    }
    cleaned = []
    seen = set()
    reader = csv.DictReader(io.StringIO(csv_text.strip()))
    for row in reader:
        report["rows_in"] += 1
        if row["id"] in seen:
            report["dropped_duplicate"] += 1
            continue
        hours = parse_float(row["hours"])
        km = parse_float(row["km"])
        age = parse_float(row["rider_age"])
        missing = row["hours"] in ("", None) or row["km"] in ("", None) or row["rider_age"] in ("", None)
        if hours is None or km is None or age is None:
            if missing:
                report["dropped_missing"] += 1
            else:
                report["dropped_invalid"] += 1
            continue
        if hours <= 0:
            report["dropped_invalid"] += 1
            continue
        seen.add(row["id"])
        cleaned.append(
            {
                "id": int(row["id"]),
                "city": row["city"],
                "hours": hours,
                "km": km,
                "rider_age": int(age),
                "pass_type": row["pass_type"],
                "returned": row["returned"] == "yes",
                "km_per_hour": km / hours,
                "age_group": age_group(int(age)),
            }
        )
    report["rows_out"] = len(cleaned)
    return cleaned, report

trips, quality = prepare(RAW_CSV)
print("quality:", quality)
print("kept ids:", [row["id"] for row in trips])
print(
    f"{quality['rows_in']} in → "
    f"missing {quality['dropped_missing']}, "
    f"invalid {quality['dropped_invalid']}, "
    f"duplicate {quality['dropped_duplicate']} → "
    f"{quality['rows_out']} out"
)

# Check 07.1
assert quality["rows_in"] == 9
assert quality["dropped_duplicate"] == 1
assert quality["dropped_missing"] == 1
assert quality["dropped_invalid"] == 1
assert quality["rows_out"] == 6
assert {row["id"] for row in trips} == {1, 3, 4, 6, 7, 8}
print("Practice 07.1 passed.")


<details>
<summary>Hint 07.1</summary>

Nine data rows. Row 2 has blank hours (missing). The second row 4 is a duplicate. Row 5 has `abc` hours (invalid). Keep ids 1, 3, 4, 6, 7, 8.

</details>


## Practice 07.2 — split, separate target, scale


In [ ]:
# Practice 07.2
FEATURE_COLUMNS = ["hours", "km", "km_per_hour"]
TARGET = "returned"


def split_rows(rows, fraction=0.8):
    ordered = sorted(rows, key=lambda row: row["id"])
    cut = int(len(ordered) * fraction)
    return ordered[:cut], ordered[cut:]


def fit_scaler(rows, columns):
    return {
        column: {
            "min": min(row[column] for row in rows),
            "max": max(row[column] for row in rows),
        }
        for column in columns
    }


def apply_scaler(rows, stats):
    output = []
    for row in rows:
        item = dict(row)
        for column, bounds in stats.items():
            span = bounds["max"] - bounds["min"]
            item[column] = 0.0 if span == 0 else (row[column] - bounds["min"]) / span
        output.append(item)
    return output


def features_and_target(rows):
    X = [[row[column] for column in FEATURE_COLUMNS] for row in rows]
    y = [row[TARGET] for row in rows]
    return X, y

train, valid = split_rows(trips, 0.8)
scaler = fit_scaler(train, FEATURE_COLUMNS)
train_scaled = apply_scaler(train, scaler)
valid_scaled = apply_scaler(valid, scaler)
X_train, y_train = features_and_target(train_scaled)
X_valid, y_valid = features_and_target(valid_scaled)

print("train", [row["id"] for row in train])
print("valid", [row["id"] for row in valid])
print("scaler", scaler)

# Check 07.2
assert [row["id"] for row in train] == [1, 3, 4, 6]
assert [row["id"] for row in valid] == [7, 8]
assert TARGET not in FEATURE_COLUMNS
assert len(X_train[0]) == 3
for column, bounds in scaler.items():
    train_values = [row[column] for row in train_scaled]
    assert min(train_values) >= 0 - 1e-9
    assert max(train_values) <= 1 + 1e-9
print("Practice 07.2 passed.")


<details>
<summary>Hint 07.2</summary>

Sort by `id`. With 6 rows, `int(6 * 0.8)` is 4, so validation is the last two ids. Fit mins and maxes on train only.

</details>


## Practice 07.3 — export and invariants


In [ ]:
# Practice 07.3
import json

export_path = DATA_DIR / "bike_trips_prepared.json"
bundle = {
    "quality": quality,
    "feature_columns": FEATURE_COLUMNS,
    "target": TARGET,
    "train_ids": [row["id"] for row in train],
    "valid_ids": [row["id"] for row in valid],
    "scaler": scaler,
}
export_path.write_text(json.dumps(bundle, indent=2), encoding="utf-8")
saved = json.loads(export_path.read_text(encoding="utf-8"))

# Check 07.3
assert saved["train_ids"] == [1, 3, 4, 6]
assert saved["valid_ids"] == [7, 8]
assert set(saved["train_ids"]).isdisjoint(saved["valid_ids"])
assert saved["quality"]["rows_out"] == 6
print("Wrote", export_path)
print("Practice 07.3 passed.")


<details>
<summary>Hint 07.3</summary>

Dump a JSON object with the quality report, ids, and scaler. Then assert train and valid ids do not overlap.

</details>


## You are ready for ML libraries when

- You can load rows, clean them (`strip`, convert, drop missing), and say what each column means.
- You can encode a category as a number and keep the target out of the feature matrix.
- You can split first and compute statistics on train only (or seed a shuffle).
- You can parse a date and read a docstring with `help` / `.__doc__`.
- You can check those promises with `assert`.

**Next:** NumPy arrays, pandas DataFrames, a plotting library, then scikit-learn's `Pipeline` and `train_test_split`. The verbs stay the same.


## Common mistakes

- Keeping duplicate id 4 or the blank-hours row.
- Scaling before splitting.
- Putting `returned` inside `X`.


## Recap


A usable ML dataset is a software product: parsed, typed, documented, split, and checked. You built that product in Python.


## Check your understanding

<details>
<summary>Does this capstone ask you to train a model?</summary>

No. Stop after a clean, split, scaled table. Libraries come next.

</details>

<details>
<summary>pandas dropna + sklearn train_test_split is closest to which step here?</summary>

Cleaning rows, then splitting before you scale.

</details>


## Solutions


<details>
<summary>Solution 07.1</summary>

```python
# Skip duplicate ids, blank hours, and non-numeric hours.
# Keep 1, 3, 4, 6, 7, 8. Derive km_per_hour = km / hours.
cleaned, report = prepare(RAW_CSV)
assert report["rows_out"] == 6
```

</details>


<details>
<summary>Solution 07.2</summary>

```python
train, valid = split_rows(trips, 0.8)
# ids [1, 3, 4, 6] then [7, 8]
scaler = fit_scaler(train, FEATURE_COLUMNS)
X_train, y_train = features_and_target(apply_scaler(train, scaler))
```

</details>


<details>
<summary>Solution 07.3</summary>

```python
bundle = {
    "quality": quality,
    "feature_columns": FEATURE_COLUMNS,
    "target": TARGET,
    "train_ids": [row["id"] for row in train],
    "valid_ids": [row["id"] for row in valid],
    "scaler": scaler,
}
export_path.write_text(json.dumps(bundle, indent=2), encoding="utf-8")
```

</details>


## Navigation

- Previous: [06_Python_Data_Workflows_for_ML.ipynb](https://github.com/ialimustufa/python-crash-course/blob/master/notebooks/06_Python_Data_Workflows_for_ML.ipynb) · [Open in Colab](https://colab.research.google.com/github/ialimustufa/python-crash-course/blob/master/notebooks/06_Python_Data_Workflows_for_ML.ipynb)
- Next: you finished the sequence. Return to the course README.
- Course home: [README.md](https://github.com/ialimustufa/python-crash-course/blob/master/README.md)
